# Räkna om Vårdbetyget ur öppna källor

Den här notebooken hämtar Nationell Patientenkät 2025 och Socialstyrelsens
väntetider direkt från källorna, räknar om Vårdbetyget med samma metod som
[vardbetyg.se/metodik](https://vardbetyg.se/metodik) och jämför med de
publicerade värdena i `units.csv`.

**Vad som räknas om helt.** Patientupplevelsen: sammanvägningen av enkätens
sex områden, rikssnittet och krympningen mot rikssnittet för vårdcentraler med
få svar. Den väger 70 procent i betyget.

**Vad som kontrolleras.** Tillgängligheten väger 30 procent. Notebooken räknar
fram varje vårdcentrals eget värde ur Socialstyrelsens publicerade andelar.
Krympningen av tillgängligheten använder antalet ärenden per månad, och det
antalet ingår aldrig i den öppna datan. Notebooken hämtar det inte. Den tar i
stället tillgänglighetspoängen ur `units.csv` och kontrollerar att den ligger
mellan vårdcentralens eget värde och rikssnittet, vilket krympningen alltid
ger.

**Avvikelser.** Hittar notebooken en avvikelse ändras inget betyg i efterhand.
Den rapporteras och rättas i nästa metodikrelease.

Krav: Python 3.10 eller senare och internet. Bara standardbiblioteket används,
så inget behöver installeras. Kör cellerna uppifrån och ned. Hämtningarna tar
någon minut.

Citera datasetet som det står i [README](README.md#så-citerar-du).

In [1]:
import csv
import html
import http.cookiejar
import json
import math
import os
import re
import time
import unicodedata
import urllib.parse
import urllib.request
from collections import defaultdict

# Metodens konstanter, samma som på vardbetyg.se/metodik.
K = 30                         # vid 30 svar väger vårdcentralens egna svar hälften
MIN_SVAR = 10                  # färre svar än så ger inget betyg
VIKT_PATIENTUPPLEVELSE = 0.7
VIKT_TILLGANGLIGHET = 0.3

# Mätningen i Nationell Patientenkät.
NPE_MATNING = "Vårdcentraler och hälsocentraler"
NPE_PERIOD = "2025"

# Väntetidernas månader. Den sista månaden ska vara densamma som exporten
# byggde på (se kodbok.md). Ändra den om du jämför med en senare export.
VANTETIDER_FORSTA_MANAD = "2025-07"
VANTETIDER_SISTA_MANAD = "2026-08"

# De publicerade värdena: units.csv bredvid notebooken, annars från repot.
PUBLICERAD_FIL = "units.csv"
PUBLICERAD_URL = "https://raw.githubusercontent.com/arpi09/vardbetyg-open-data/main/units.csv"

ANVANDARAGENT = "vardbetyg-open-data-notebook (https://github.com/arpi09/vardbetyg-open-data)"

## 1. Hämta Nationell Patientenkät

Patientenkätens öppna API ([resultat.patientenkat.se](https://resultat.patientenkat.se))
ger varje vårdcentrals resultat per område, antal svarande och status. Två
anrop räcker: `measurementStat` listar enheterna och `dimensionScores` ger
resultaten.

In [2]:
NPE_BAS = "https://resultat.patientenkat.se/gss-results/rest/stats"


def hamta_npe(sokvag, **parametrar):
    parametrar["audience"] = "public"
    url = (f"{NPE_BAS}/{urllib.parse.quote(NPE_MATNING)}/{sokvag}"
           f"?{urllib.parse.urlencode(parametrar)}")
    fraga = urllib.request.Request(url, headers={"User-Agent": ANVANDARAGENT})
    with urllib.request.urlopen(fraga, timeout=300) as svar:
        return json.loads(svar.read().decode("utf-8"))


matning = hamta_npe("measurementStat", periods=NPE_PERIOD)
resultat = hamta_npe("public/dimensionScores", periods=NPE_PERIOD)

organisationer = {}
for struktur in matning["orgStructures"]:
    for org in struktur["organisations"]:
        organisationer.setdefault(org["id"], org)

vardcentraler = [
    (organisationer[rad["orgIds"]], rad)
    for rad in resultat
    if organisationer.get(rad["orgIds"], {}).get("type") == "Hälso-/vårdcentral"
]
print(f"Vårdcentraler i patientenkäten {NPE_PERIOD}: {len(vardcentraler)}")
print(f"Med redovisat resultat (status ok): {sum(r['status'] == 'ok' for _, r in vardcentraler)}")

Vårdcentraler i patientenkäten 2025: 1210
Med redovisat resultat (status ok): 1201


## 2. Patientupplevelsen, med krympning mot rikssnittet

För varje vårdcentral:

- **Sammanvägning** = medelvärdet av områdenas poäng (0 till 100), alla
  områden väger lika.
- **Rikssnitt** = medelvärdet över områdena av alla vårdcentralers medel per
  område.
- **Vikt** = n / (n + 30), där n är antal svarande.
- **Krympt poäng** = vikt × sammanvägning + (1 − vikt) × rikssnitt.

Färre än 10 svarande ger inget betyg.

In [3]:
def normalisera_hsa(hsa_id):
    """Samma HSA-id skrivs ibland med och ibland utan sekelsiffrorna 16."""
    hsa_id = (hsa_id or "").strip().upper()
    return re.sub(r"^SE16(?=\d{10}-)", "SE", hsa_id)


per_omrade = defaultdict(list)
for org, rad in vardcentraler:
    if rad["status"] != "ok":
        continue
    for omrade in rad.get("dimensions") or []:
        if omrade.get("score") is not None:
            per_omrade[omrade["name"]].append(omrade["score"])

omradesmedel = {namn: sum(v) / len(v) for namn, v in per_omrade.items()}
rikssnitt_npe = sum(omradesmedel.values()) / len(omradesmedel)

enheter = {}
for org, rad in vardcentraler:
    svarande = (rad.get("invitees") or {}).get("answered")
    poang = {o["name"]: o["score"] for o in (rad.get("dimensions") or []) if o.get("score") is not None}
    enhet = {"hsa": normalisera_hsa(org.get("hsaId")), "svarande": svarande,
             "tillfragade": (rad.get("invitees") or {}).get("invited"),
             "omraden": poang, "sammanvagning": None, "patientupplevelse": None}
    if rad["status"] == "ok" and poang and svarande is not None and svarande >= MIN_SVAR:
        sammanvagning = sum(poang.values()) / len(poang)
        vikt = svarande / (svarande + K)
        enhet["sammanvagning"] = sammanvagning
        enhet["patientupplevelse"] = vikt * sammanvagning + (1 - vikt) * rikssnitt_npe
    enheter[org["id"]] = enhet

enheter_per_hsa = {e["hsa"]: e for e in enheter.values() if e["hsa"]}

print("Medel per område:")
for namn, medel in sorted(omradesmedel.items()):
    print(f"  {namn:<22} {medel:6.2f}")
print(f"Rikssnitt för patientupplevelsen: {rikssnitt_npe:.3f}")
print(f"Vårdcentraler med betyg: {sum(e['patientupplevelse'] is not None for e in enheter.values())}")

Medel per område:
  Bemötande               83.90
  Delaktighet             80.39
  Information             74.82
  Kontinuitet             70.44
  Tillgänglighet          80.19
  Vård och behandling     79.33
Rikssnitt för patientupplevelsen: 78.178
Vårdcentraler med betyg: 1201


## 3. Hämta väntetidernas andelar från Socialstyrelsen

Tillgängligheten bygger på två andelar i Socialstyrelsens statistikdatabas
([sdb.socialstyrelsen.se/vantetider-primarvard](https://sdb.socialstyrelsen.se/vantetider-primarvard/)):

- andel med medicinsk bedömning inom 3 dagar (måttet `InomTreAndel`),
- andel besvarade samtal (måttet `BesvaradeAndel`, alla samtal).

Databasen är ett äldre webbformulär utan API. Koden fyller i formuläret som en
webbläsare gör och laddar ner samma CSV som knappen i formuläret. Bara de två
andelarna hämtas, aldrig antalen. Socialstyrelsen har bytt id på måtten
tidigare, så varje fil kontrolleras mot sin rubrik.

In [4]:
SDB_BAS = "https://sdb.socialstyrelsen.se/vantetider-primarvard/"
ANDELAR = [
    {"fil": "bedomning", "fordelning": "VARDCENTRAL", "matt": "InomTreAndel",
     "rubrik": "Andel (procent) med väntetid inom 3 dagar"},
    {"fil": "samtal", "fordelning": "TELECHATT", "matt": "BesvaradeAndel",
     "rubrik": "Andel (procent) besvarade samtal"},
]
TOMMA_FALT = [
    "AGI_RES", "ARMANAD_IND_RES", "AR_MANAD_RES", "AR_RES", "BESOKSFORM_RES", "DELOMRADE_RES",
    "FORDELNING_RES", "IND_RES", "KON_RES", "MANAD_RES", "MATT_TELECHATT_RES", "OMR_ANTALRADER_RES",
    "OMR_RES", "PVTYP_RES", "REGION_RES", "VARDCENTRAL_RES", "YRKE_RES", "haOMR", "hvOMR", "hvOMR2",
    "vAGI", "vAR", "vARMANAD_IND", "vBESOKSFORM", "vDELOMRADE", "vKON", "vYRKE",
]
JSTEXT = ("Resultatet kan max bestå av 70 000 celler|Följande har inte valts: |fördelning|kontaktstatus|"
          "fas|år|månad|mått|region|kön|ålder|vårdgivare|specialistvårdsenhet|sjukhus|specialitet|"
          "aktivitet|verksamhetsområde|besöksform|vårdcentral|yrke|typ")

oppnare = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))
oppnare.addheaders = [("User-Agent", ANVANDARAGENT)]


def sdb_post(url, falt, referer):
    data = urllib.parse.urlencode(falt).encode("utf-8")
    fraga = urllib.request.Request(url, data=data, headers={
        "Content-Type": "application/x-www-form-urlencoded", "Referer": referer})
    with oppnare.open(fraga, timeout=300) as svar:
        return svar.headers.get("Content-Type", ""), svar.read()


def dolda_falt(sida):
    falt = {}
    for namn, varde in re.findall(r"<input[^>]*type=['\"]hidden['\"][^>]*name=['\"]([^'\"]+)['\"][^>]*value=['\"]([^'\"]*)['\"]", sida, re.I):
        falt[namn] = html.unescape(varde)
    for varde, namn in re.findall(r"<input[^>]*type=['\"]hidden['\"][^>]*value=['\"]([^'\"]*)['\"][^>]*name=['\"]([^'\"]+)['\"]", sida, re.I):
        falt.setdefault(namn, html.unescape(varde))
    return falt


def lista(sida, ident):
    traff = re.search(r"id='%s' name='[^']+' value='([^']*)'" % ident, sida)
    if traff is None:
        raise RuntimeError(f"Hittade inte {ident} på urvalssidan. Har formuläret ändrats?")
    return traff.group(1).split("|")


def v(varden):
    return "".join(f";{x};" for x in varden)


with oppnare.open(SDB_BAS, timeout=300) as svar:
    urvalssida = svar.read().decode("utf-8", errors="replace")
regionval = re.search(r"<select[^>]*id=['\"]REGION['\"][\s\S]*?</select>", urvalssida, re.I).group(0)
regioner = re.findall(r"<option[^>]*value=['\"]([^'\"]*)['\"]", regionval)

forsta_ar, forsta_man = (int(x) for x in VANTETIDER_FORSTA_MANAD.split("-"))
sista_ar, sista_man = (int(x) for x in VANTETIDER_SISTA_MANAD.split("-"))
ar = [str(a) for a in range(sista_ar, forsta_ar - 1, -1)]
manader = [str(m) for m in range(1, 13)]

sdb_enheter = {}   # (regionkod, etikett) -> HSA-id, ur urvalssidans enhetslistor
sdb_csv = {}
for andel in ANDELAR:
    prefix = "RSS4" if andel["fordelning"] == "TELECHATT" else "RSS3"
    idn = lista(urvalssida, f"{prefix}_SJUKHUS")
    namn = lista(urvalssida, f"{prefix}_SJUKHUS_TEXT")
    regionkoder = lista(urvalssida, f"{prefix}_REGION")
    enhets_idn = []
    for i, hsa_id in enumerate(idn):
        if hsa_id in ("", "0"):
            continue
        sdb_enheter.setdefault((regionkoder[i], namn[i].strip()), hsa_id)
        if hsa_id not in enhets_idn:
            enhets_idn.append(hsa_id)

    falt = [("FORDELNING", andel["fordelning"]), ("vFORDELNING", v([andel["fordelning"]]))]
    if andel["fordelning"] == "TELECHATT":
        falt += [("MATT_TELECHATT", andel["matt"]), ("vMATT_TELECHATT", v([andel["matt"]])),
                 ("PVTYP", "0"), ("vPVTYP", v(["0"])), ("vIND", "")]
    else:
        falt += [("IND", andel["matt"]), ("vIND", v([andel["matt"]])),
                 ("vMATT_TELECHATT", ""), ("vPVTYP", "")]
    falt += [("AR_MANAD", a) for a in ar] + [("vAR_MANAD", v(ar))]
    falt += [("MANAD", m) for m in manader] + [("vMANAD", v(manader))]
    falt += [("REGION", r) for r in regioner] + [("vREGION", v(regioner))]
    falt += [("VARDCENTRAL", e) for e in enhets_idn] + [("vVARDCENTRAL", v(enhets_idn))]
    falt += [("TYP", "TABELL"), ("AR_MIN", str(forsta_ar)), ("MANAD_MIN", str(forsta_man)),
             ("AR_MAX", str(sista_ar)), ("MANAD_MAX", str(sista_man)),
             ("OMR_ANTALRADER", str(len(idn))), ("Sokord_VARDCENTRAL", ""), ("sprak", ""),
             ("clientScreenWidth", "950"), ("clientScreenHeight", "609"), ("JSText", JSTEXT)]
    falt += [(f, "") for f in TOMMA_FALT]

    _, svar = sdb_post(SDB_BAS + "resultat.aspx", falt, SDB_BAS)
    dolda = dolda_falt(svar.decode("utf-8", errors="replace"))
    if "__VIEWSTATE" not in dolda:
        raise RuntimeError("Socialstyrelsen avvisade urvalet.")
    tillbaka = [(k, x) for k, x in dolda.items() if k not in ("__EVENTTARGET", "__EVENTARGUMENT")]
    tillbaka += [("__EVENTTARGET", "ctl00$ph1$lbCSV"), ("__EVENTARGUMENT", "")]
    typ, data = sdb_post(SDB_BAS + "resultat.aspx", tillbaka, SDB_BAS + "resultat.aspx")
    text = data.decode("utf-8-sig")
    forsta_raden = text.splitlines()[0]
    if andel["rubrik"] not in forsta_raden:
        raise RuntimeError(f"Fel mått: rubriken är {forsta_raden!r}, väntade {andel['rubrik']!r}.")
    sdb_csv[andel["fil"]] = text
    print(f"{forsta_raden}: {len(text.splitlines())} rader")
    time.sleep(1)   # var snäll mot servern

Väntetider till primärvård, Andel (procent) med väntetid inom 3 dagar: 1670 rader
Väntetider till primärvård, Andel (procent) besvarade samtal, Typ: Alla samtal: 1108 rader


## 4. Tillgänglighetens värde per vårdcentral

Varje rad i Socialstyrelsens fil kopplas till en vårdcentral i patientenkäten
på samma sätt som sajten gör det:

1. via HSA-id: urvalssidans enhetslistor ger HSA-id för varje etikett,
2. annars via region och namn, med vårdcentralens namn som det står i
   `units.csv`.

Ett observerat värde går före en tom månad när två rader hamnar på samma
vårdcentral. Värdet per andel är den senaste månaden med ett värde, och
vårdcentralens tillgänglighet är medelvärdet av de två andelarna.
Rikssnittet för tillgängligheten är medelvärdet över vårdcentralerna.

En del vårdcentraler har registrerats om hos Socialstyrelsen med ett nytt
HSA-id. Sajten kopplar dem med en egen kopplingslista som inte ingår i den
öppna datan. Notebooken redovisar dem som ofullständigt kopplade.

In [5]:
MANADSNAMN = {"januari": "01", "februari": "02", "mars": "03", "april": "04", "maj": "05",
              "juni": "06", "juli": "07", "augusti": "08", "september": "09", "oktober": "10",
              "november": "11", "december": "12"}
REGIONKODER = {"1": "Stockholm", "3": "Uppsala", "4": "Södermanland", "5": "Östergötland",
               "6": "Jönköping", "7": "Kronoberg", "8": "Kalmar", "9": "Gotland", "10": "Blekinge",
               "12": "Skåne", "13": "Halland", "14": "Västra Götaland", "17": "Värmland",
               "18": "Örebro", "19": "Västmanland", "20": "Dalarna", "21": "Gävleborg",
               "22": "Västernorrland", "23": "Jämtland Härjedalen", "24": "Västerbotten",
               "25": "Norrbotten"}


# Osynliga tecken som Socialstyrelsen strör i namnen (mjukt bindestreck,
# nollbreda mellanslag) och skiljetecken som blir mellanslag vid jämförelsen.
OSYNLIGA_TECKEN = "[" + "".join(map(chr, (0xAD, 0x200B, 0x200C, 0x200D, 0xFEFF))) + "]"
SKILJETECKEN = "[.,/" + chr(0x2013) + chr(0x2014) + "-]"


def normalisera_namn(namn):
    namn = unicodedata.normalize("NFC", namn)
    namn = re.sub(OSYNLIGA_TECKEN, "", namn).lower()
    namn = re.sub(SKILJETECKEN, " ", namn)
    namn = re.sub(r"\s+", " ", namn).strip()
    return re.sub(r"\s+ab$", "", namn)


def las_publicerade():
    if os.path.exists(PUBLICERAD_FIL):
        with open(PUBLICERAD_FIL, encoding="utf-8", newline="") as fil:
            return list(csv.DictReader(fil))
    with urllib.request.urlopen(urllib.request.Request(PUBLICERAD_URL, headers={"User-Agent": ANVANDARAGENT}), timeout=300) as svar:
        return list(csv.DictReader(svar.read().decode("utf-8").splitlines()))


publicerade = las_publicerade()

# HSA-id per etikett (region, normaliserat namn). En etikett med två olika id kopplas inte.
etikett_hsa, tvetydiga = {}, set()
for (regionkod, etikett), hsa_id in sdb_enheter.items():
    region = REGIONKODER.get(regionkod)
    if region is None:
        continue
    nyckel = (normalisera_namn(region), normalisera_namn(etikett))
    if nyckel in etikett_hsa and etikett_hsa[nyckel] != normalisera_hsa(hsa_id):
        tvetydiga.add(nyckel)
    else:
        etikett_hsa[nyckel] = normalisera_hsa(hsa_id)
for nyckel in tvetydiga:
    del etikett_hsa[nyckel]

# Namnreserven: vårdcentralens namn som det står i units.csv.
namnindex = defaultdict(list)
for rad in publicerade:
    namnindex[(normalisera_namn(rad["region"]), normalisera_namn(rad["name"]))].append(rad)


def koppla(region, etikett):
    hsa_id = etikett_hsa.get((normalisera_namn(region), normalisera_namn(etikett)))
    if hsa_id is not None and hsa_id in enheter_per_hsa:
        return hsa_id, "hsa"
    namn, ort = (etikett.rsplit(",", 1) + [None])[:2] if "," in etikett else (etikett, None)
    kandidater = namnindex.get((normalisera_namn(region), normalisera_namn(namn.strip())), [])
    if len(kandidater) == 1:
        return normalisera_hsa(kandidater[0]["hsa_id"]), "namn"
    if len(kandidater) > 1 and ort:
        for kandidat in kandidater:
            if normalisera_namn(kandidat["kommun"]) == normalisera_namn(ort.strip()):
                return normalisera_hsa(kandidat["hsa_id"]), "namn"
    return None, None


def las_andel(text):
    rader = [r for r in text.splitlines() if r]
    arsrad, rubrikrad = rader[1].split(";"), rader[2].split(";")
    kolumner = [(i, f"{arsrad[i].strip()}-{MANADSNAMN[c.strip().lower()]}")
                for i, c in enumerate(rubrikrad) if c.strip().lower() in MANADSNAMN and arsrad[i].strip()]
    for rad in rader[3:]:
        celler = rad.split(";")
        if len(celler) < 2 or not celler[1].strip() or celler[1].strip().lower() == "alla vårdcentraler":
            continue
        for i, period in kolumner:
            cell = (celler[i] if i < len(celler) else "").strip().replace(",", ".")
            try:
                varde = float(cell)
            except ValueError:
                varde = None
            yield celler[0].strip(), celler[1].strip(), period, varde


senaste = defaultdict(dict)   # hsa -> andel -> (period, värde)
for andel, text in sdb_csv.items():
    celler = {}
    for region, etikett, period, varde in las_andel(text):
        hsa_id, via = koppla(region, etikett)
        if hsa_id is None:
            continue
        rang = (2 if varde is not None else 0) + (1 if via == "hsa" else 0)
        if (hsa_id, period) not in celler or rang > celler[(hsa_id, period)][0]:
            celler[(hsa_id, period)] = (rang, varde)
    for (hsa_id, period), (_, varde) in celler.items():
        if varde is not None and (andel not in senaste[hsa_id] or period > senaste[hsa_id][andel][0]):
            senaste[hsa_id][andel] = (period, varde)

tillganglighet = {h: sum(x for _, x in a.values()) / len(a) for h, a in senaste.items() if a}
rikssnitt_tillganglighet = sum(tillganglighet.values()) / len(tillganglighet)
print(f"Vårdcentraler med ett eget tillgänglighetsvärde: {len(tillganglighet)}")
print(f"Rikssnitt för tillgängligheten (de kopplade vårdcentralerna): {rikssnitt_tillganglighet:.2f}")

Vårdcentraler med ett eget tillgänglighetsvärde: 1142
Rikssnitt för tillgängligheten (de kopplade vårdcentralerna): 88.42


## 5. Jämför med de publicerade värdena

Jämförelsen görs på det publicerade, avrundade värdet (en decimal):

- **Underlaget**: områdenas poäng och antal svarande ska vara desamma som i
  patientenkäten.
- **Patientupplevelsen** och sammanvägningen före krympning ska stämma exakt.
- **Vårdbetyget** räknas som 0,7 × patientupplevelsen + 0,3 ×
  tillgängligheten, eller bara patientupplevelsen när väntetider saknas.
  Tillgänglighetspoängen i `units.csv` är avrundad till en decimal, så
  betyget kan skilja en tiondel. Då kontrolleras att det publicerade betyget
  ryms inom avrundningen av den poängen.
- **Tillgängligheten** ska ligga mellan vårdcentralens eget värde och
  rikssnittet.

In [6]:
def tal(text):
    return float(text) if text not in ("", None) else None


def avrunda(x):
    """En decimal, halvor uppåt, som sajten avrundar."""
    return None if x is None else math.floor(x * 10 + 0.5) / 10


# Rikssnittet för tillgängligheten går att läsa ur units.csv: för vårdcentraler
# med väntetider är vardbetyg_prior = 0,7 × rikssnittet för patientupplevelsen
# + 0,3 × rikssnittet för tillgängligheten, avrundat till en decimal.
prior_med_tg = {tal(r["vardbetyg_prior"]) for r in publicerade if r["tillganglighet_score"] and r["vardbetyg_prior"]}
assert len(prior_med_tg) == 1, prior_med_tg
blandat = prior_med_tg.pop()
tg_rikssnitt_lag = (blandat - 0.05 - VIKT_PATIENTUPPLEVELSE * rikssnitt_npe) / VIKT_TILLGANGLIGHET
tg_rikssnitt_hog = (blandat + 0.05 - VIKT_PATIENTUPPLEVELSE * rikssnitt_npe) / VIKT_TILLGANGLIGHET
print(f"Rikssnittet för tillgängligheten enligt units.csv: {tg_rikssnitt_lag:.2f} till {tg_rikssnitt_hog:.2f}")
print(f"Notebookens eget, ur de kopplade vårdcentralerna: {rikssnitt_tillganglighet:.2f}")
print()

granskning = defaultdict(int)
avvikelser = []
ofullstandiga = []
tiondelar = []

for rad in publicerade:
    hsa_id = normalisera_hsa(rad["hsa_id"])
    enhet = enheter_per_hsa.get(hsa_id)
    if enhet is None:
        avvikelser.append((rad["slug"], "finns inte i patientenkäten"))
        continue
    granskning["vårdcentraler i units.csv"] += 1

    # Underlaget ska vara patientenkätens egna tal.
    for omrade, poang in enhet["omraden"].items():
        publicerad = tal(rad.get(f"dim:{omrade}"))
        if publicerad is not None and abs(avrunda(poang) - publicerad) > 1e-9:
            avvikelser.append((rad["slug"], f"området {omrade}: {poang} mot {publicerad}"))
    if tal(rad["answered"]) != enhet["svarande"]:
        avvikelser.append((rad["slug"], "antal svarande skiljer"))

    betyg = tal(rad["vardbetyg"])
    if enhet["patientupplevelse"] is None:
        if betyg is not None:
            avvikelser.append((rad["slug"], "publicerat betyg trots för få svar"))
        else:
            granskning["för få svar, inget betyg (stämmer)"] += 1
        continue
    if betyg is None:
        avvikelser.append((rad["slug"], "betyg saknas fast underlaget räcker"))
        continue

    if abs(avrunda(enhet["sammanvagning"]) - tal(rad["vardbetyg_raw_composite"])) < 1e-9:
        granskning["sammanvägning före krympning exakt"] += 1
    else:
        avvikelser.append((rad["slug"], "sammanvägningen skiljer"))
    if abs(avrunda(enhet["patientupplevelse"]) - tal(rad["patientupplevelse_score"])) < 1e-9:
        granskning["patientupplevelse efter krympning exakt"] += 1
    else:
        avvikelser.append((rad["slug"], "patientupplevelsen skiljer"))

    pu = enhet["patientupplevelse"]
    tg = tal(rad["tillganglighet_score"])
    if tg is None:
        omraknat, lag, hog = pu, pu, pu
    else:
        omraknat = VIKT_PATIENTUPPLEVELSE * pu + VIKT_TILLGANGLIGHET * tg
        lag = VIKT_PATIENTUPPLEVELSE * pu + VIKT_TILLGANGLIGHET * (tg - 0.05)
        hog = VIKT_PATIENTUPPLEVELSE * pu + VIKT_TILLGANGLIGHET * (tg + 0.05)
    if abs(avrunda(omraknat) - betyg) < 1e-9:
        granskning["Vårdbetyget exakt"] += 1
    elif lag - 0.05 - 1e-9 <= betyg <= hog + 0.05 + 1e-9:
        granskning["Vårdbetyget en tiondel ifrån, inom avrundningen"] += 1
        tiondelar.append(rad["slug"])
    else:
        avvikelser.append((rad["slug"], f"Vårdbetyget {omraknat:.2f} mot publicerat {betyg}"))

    if tg is not None:
        granskning["med tillgänglighetspoäng"] += 1
        eget = tillganglighet.get(hsa_id)
        if eget is None:
            ofullstandiga.append((rad["slug"], "ingen rad hos Socialstyrelsen kunde kopplas"))
            continue
        lagst = min(eget, tg_rikssnitt_lag) - 0.05
        hogst = max(eget, tg_rikssnitt_hog) + 0.05
        if lagst - 1e-9 <= tg <= hogst + 1e-9:
            granskning["tillgänglighet mellan eget värde och rikssnitt"] += 1
        else:
            perioder = [p for p, _ in senaste[hsa_id].values()]
            if min(perioder) < VANTETIDER_SISTA_MANAD:
                ofullstandiga.append((rad["slug"], f"senaste värde {min(perioder)}, troligen omregistrerad"))
            else:
                avvikelser.append((rad["slug"], f"tillgängligheten {tg} ligger utanför {eget:.1f} till rikssnittet"))

# Utan väntetider är vardbetyg_prior rikssnittet för patientupplevelsen.
priorer = {tal(r["vardbetyg_prior"]) for r in publicerade if r["tillganglighet_score"] == "" and r["vardbetyg_prior"]}
granskning_prior = priorer == {avrunda(rikssnitt_npe)}

for namn, antal in granskning.items():
    print(f"{namn:<52} {antal:>5}")
print()
print(f"Rikssnittet för patientupplevelsen ({rikssnitt_npe:.2f}) stämmer med units.csv: {granskning_prior}")

Rikssnittet för tillgängligheten enligt units.csv: 88.08 till 88.42
Notebookens eget, ur de kopplade vårdcentralerna: 88.42

vårdcentraler i units.csv                             1192
sammanvägning före krympning exakt                    1192
patientupplevelse efter krympning exakt               1192
Vårdbetyget exakt                                     1105
med tillgänglighetspoäng                              1153
tillgänglighet mellan eget värde och rikssnitt        1122
Vårdbetyget en tiondel ifrån, inom avrundningen         87

Rikssnittet för patientupplevelsen (78.18) stämmer med units.csv: True


In [7]:
print(f"Avvikelser: {len(avvikelser)}")
for slug, orsak in avvikelser:
    print(f"  {slug}: {orsak}")
print()
print(f"Ofullständigt kopplade hos Socialstyrelsen (inte avvikelser i betyget): {len(ofullstandiga)}")
for slug, orsak in ofullstandiga:
    print(f"  {slug}: {orsak}")
print()
if not avvikelser:
    print("Resultat: notebooken återskapar de publicerade betygen inom avrundning.")
else:
    print("Resultat: avvikelser hittades. Rapportera dem, betygen rättas i nästa metodikrelease.")

Avvikelser: 0

Ofullständigt kopplade hos Socialstyrelsen (inte avvikelser i betyget): 31
  allmanmedicinsk-mottagning-vardcentralen-grums: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinsk-mottagning-vardcentralen-nysater: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-josse-vardcentral-arvika: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-vardcentralen-eda: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-vardcentralen-filipstad: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-vardcentralen-forshaga: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-vardcentralen-gripen-karlstad: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-vardcentralen-hagfors-eksharad: ingen rad hos Socialstyrelsen kunde kopplas
  allmanmedicinska-mottagningen-vardcentralen-herrhagen-karlstad: ingen rad hos Socialstyrelsen kunde koppl